# Simulations In Three Dimensions Example

Ported from: k-Wave/examples/example_ivp_3D_simulation.m

Simulates the pressure field generated by an initial pressure distribution
(two balls of equal magnitude) within a three-dimensional heterogeneous
propagation medium. The left half of the domain has a higher sound speed
(1800 m/s vs 1500 m/s) and the lower three-quarters has a higher density
(1200 kg/m^3 vs 1000 kg/m^3).

It builds on the Homogeneous Propagation Medium and Heterogeneous
Propagation Medium examples.

In [ ]:
!pip install k-wave-python

In [ ]:
import numpy as np

from kwave.data import Vector
from kwave.kgrid import kWaveGrid
from kwave.kmedium import kWaveMedium
from kwave.ksensor import kSensor
from kwave.ksource import kSource
from kwave.kspaceFirstOrder import kspaceFirstOrder
from kwave.utils.mapgen import make_ball

In [ ]:
def setup():
    """Set up the simulation physics (grid, medium, source).

    Returns:
        tuple: (kgrid, medium, source)
    """

    # create the computational grid
    Nx = 64  # number of grid points in the x direction
    Ny = 64  # number of grid points in the y direction
    Nz = 64  # number of grid points in the z direction
    dx = 0.1e-3  # grid point spacing in the x direction [m]
    dy = 0.1e-3  # grid point spacing in the y direction [m]
    dz = 0.1e-3  # grid point spacing in the z direction [m]
    kgrid = kWaveGrid(Vector([Nx, Ny, Nz]), Vector([dx, dy, dz]))

    # define the properties of the propagation medium
    c = 1500 * np.ones((Nx, Ny, Nz))  # [m/s]
    c[:32, :, :] = 1800  # MATLAB: 1:Nx/2 (1-based) -> :32 (0-based)
    rho = 1000 * np.ones((Nx, Ny, Nz))  # [kg/m^3]
    rho[:, 15:, :] = 1200  # MATLAB: Ny/4:end = 16:64 (1-based) -> 15: (0-based)
    medium = kWaveMedium(sound_speed=c, density=rho)

    # create initial pressure distribution using make_ball
    # -- ball 1 --
    ball_magnitude = 10  # [Pa]
    ball_1_x_pos = 38  # [grid points, 1-based]
    ball_1_y_pos = 32  # [grid points, 1-based]
    ball_1_z_pos = 32  # [grid points, 1-based]
    ball_1_radius = 5  # [grid points]
    ball_1 = ball_magnitude * make_ball(
        Vector([Nx, Ny, Nz]),
        Vector([ball_1_x_pos, ball_1_y_pos, ball_1_z_pos]),
        ball_1_radius,
    )

    # -- ball 2 --
    ball_2_x_pos = 20  # [grid points, 1-based]
    ball_2_y_pos = 20  # [grid points, 1-based]
    ball_2_z_pos = 20  # [grid points, 1-based]
    ball_2_radius = 3  # [grid points]
    ball_2 = ball_magnitude * make_ball(
        Vector([Nx, Ny, Nz]),
        Vector([ball_2_x_pos, ball_2_y_pos, ball_2_z_pos]),
        ball_2_radius,
    )

    source = kSource()
    source.p0 = (ball_1 + ball_2).astype(float)

    # set time array (pass full c array -- makeTime uses max internally for dt)
    kgrid.makeTime(c)

    return kgrid, medium, source

In [ ]:
def run(backend="python", device="cpu", quiet=True):
    """Run with the original Cartesian sensor.

    The MATLAB original uses a Cartesian sensor mask along the y=22*dy plane.
    For simplicity the Python port uses a full-grid binary sensor, which is
    the format required for parity testing against MATLAB references.

    Returns:
        dict: Simulation results with key 'p_final' (p omitted to save memory).
    """
    kgrid, medium, source = setup()

    Nx, Ny, Nz = 64, 64, 64

    # full-grid binary sensor (matches MATLAB ref format)
    # only record p_final to keep memory manageable (full p would be ~880 MB)
    sensor = kSensor(mask=np.ones((Nx, Ny, Nz), dtype=bool))
    sensor.record = ["p_final"]

    return kspaceFirstOrder(
        kgrid,
        medium,
        source,
        sensor,
        backend=backend,
        device=device,
        quiet=quiet,
        pml_inside=True,
    )

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    result = run(quiet=False)
    p_final = np.asarray(result["p_final"])

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # plot the final pressure in three orthogonal slices
    Nx_pml = p_final.shape[0]  # grid size after PML stripping
    mid = Nx_pml // 2

    ax = axes[0]
    im = ax.imshow(p_final[:, :, mid].T, cmap="RdBu_r")
    ax.set_xlabel("x [grid points]")
    ax.set_ylabel("y [grid points]")
    ax.set_title(f"p_final (z={mid} slice)")
    fig.colorbar(im, ax=ax)

    ax = axes[1]
    im = ax.imshow(p_final[:, mid, :].T, cmap="RdBu_r")
    ax.set_xlabel("x [grid points]")
    ax.set_ylabel("z [grid points]")
    ax.set_title(f"p_final (y={mid} slice)")
    fig.colorbar(im, ax=ax)

    ax = axes[2]
    im = ax.imshow(p_final[mid, :, :].T, cmap="RdBu_r")
    ax.set_xlabel("y [grid points]")
    ax.set_ylabel("z [grid points]")
    ax.set_title(f"p_final (x={mid} slice)")
    fig.colorbar(im, ax=ax)

    fig.suptitle("Simulations In Three Dimensions Example")
    fig.tight_layout()
    plt.show()